# Day 6: Session 6A - The Join Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6a_joining_data.html)

Date: 09/08/2026

### write the join pattern, pd.merge(left, right, on='key'), and explain what the key does

### choose between how='inner' and how='left', and explain what each one does to your row count

### check a merge after you run it, and know which numbers to compare to ensure things have gone according to plan

### join two tables whose key columns have different names, using left_on= and right_on=

### explain why the rows a join drops are rarely a random sample of your data

In [ ]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/openaq_goleta_measurments.csv'
goleta = pd.read_csv(url)

goleta['parameter'].value_counts()

In [ ]:
# Let's create two dataframes using filters:

o3 = goleta[goleta['parameter'] == 'o3']
pm25 = goleta[goleta['parameter'] == 'pm25']

In [ ]:
# Let's filter our columns to the ones we want
# provide a list of columns into our selection brackets
o3 = o3[['datetimeLocal', 'value']]
o3 = o3.rename(columns={'value': 'o3_ppm'})

pm25 = pm25[['datetimeLocal', 'value']]
pm25 = pm25.rename(columns={'value': 'pm25_ugm3'})

o3.head()

# 🐍 Renaming before a join is not just cosmetic. 
# If both tables arrive at the join with a column called value, 
# pandas cannot keep them both under that name, 
# so it renames them value_x and value_y and leaves you to work out which is which. 
# Naming your columns first costs you two lines now, but saves a lot of headache later…

### **Join functions**
![alt text](https://eds-217-essential-python.github.io/course-materials/images/join_types_venn.svg)

**Outer joins** 

**Inner joins**

**Left/Right joins** are *opinionated* joins, in which you choose which dataframe to prioritize the data you want to keep and sacrifice.

### Visualization of joins
![alt text](https://eds-217-essential-python.github.io/course-materials/images/join_types_keyaxis.svg)

#### Ask yourself:
Do I want all the data where both occur?

Do I want only the data where either occur?

Do I want specific data from where one dataset occurs?

In [ ]:
# pd.merge(left, right, on='key')
#            ↑     ↑         ↑
#        the two tables    the column they have in common

In [ ]:
# Merge o3 and pm25 by datetimeLocal
paired = pd.merge(o3, pm25, on='datetimeLocal') 
#inner join is the DEFAULT function when not specified
paired.head()

,datetimeLocal,o3_ppm,pm25_ugm3
0,2024-07-11T18:00:00-07:00,0.025,3.0
1,2024-07-11T19:00:00-07:00,0.028,8.0
2,2024-07-11T20:00:00-07:00,0.029,6.0
3,2024-07-11T21:00:00-07:00,0.027,4.0
4,2024-07-11T22:00:00-07:00,0.026,9.0


In [36]:
inner = pd.merge(o3, pm25, on='datetimeLocal', how='inner')
# Second Rule of Python
# always be explicit about the "how"
inner.shape

(704, 3)

## Rules of Python 
#### (Run "import this")

1. Beautiful is better than ugly.

2. **Explicit is better than implicit.**

3. Simple is better than complex.

4. Complex is better than complicated.

5. Flat is better than nested.

6. Sparse is better than dense.

7. Readability counts.

8. Special cases aren't special enough to break the rules.

9. Although practicality beats purity.

10. Errors should never pass silently. Unless explicitly silenced.

11. In the face of ambiguity, refuse the temptation to guess.

12. There should be one-- and preferably only one --obvious way to do it. Although that way may not be obvious at first unless you're Dutch.

13. Now is better than never. Although never is often better than *right* now.

14. If the implementation is hard to explain, it's a bad idea.

15. If the implementation is easy to explain, it may be a good idea.

16. Namespaces are one honking great idea -- let's do more of those!


In [37]:
left = pd.merge(o3, pm25, on='datetimeLocal', how='left')

left.shape

(711, 3)

In [43]:
left.isnull().sum()
# 7 missing pm25 values

datetimeLocal    0
o3_ppm           0
pm25_ugm3        7
dtype: int64

In [44]:
## Don't use this very often
right = pd.merge(o3, pm25, on='datetimeLocal', how='right')

print(right.shape)
print(right.isnull().sum())
# 30 missing o3 values

(734, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         0
dtype: int64


In [ ]:
outer = pd.merge(o3, pm25, on='datetimeLocal', how='outer')

print(outer.shape)
print(outer.isnull().sum())
# outer always outputs the largest dataframe
# BUT both columns are missing data where one column had data and the other didn't
# Sparse is better than dense.

(741, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         7
dtype: int64


In [53]:
pm10 = goleta[goleta['parameter'] == 'pm10']
pm10 = pm10[['datetimeLocal', 'value']]
pm10 = pm10.rename(columns={'value': 'pm10_ug3'})


In [56]:
inner_pm10 = pd.merge(pm10, pm25, on='datetimeLocal', how='inner')
inner_pm10.shape

(512, 3)

In [57]:
left_pm10 = pd.merge(pm10, pm25, on='datetimeLocal', how='left')
left_pm10.shape

(517, 3)

In [64]:
missing_o3 = right[right['o3_ppm'].isnull()].copy()

missing_o3['datetimeLocal'].str[11:13].value_counts()

datetimeLocal
03    30
Name: count, dtype: int64

In [62]:
o3['datetimeLocal'].str[11:13].value_counts().sort_index()

datetimeLocal
00    31
01    30
02    30
04    31
05    31
06    31
07    31
08    31
09    31
10    31
11    31
12    31
13    31
14    31
15    31
16    31
17    31
18    31
19    31
20    31
21    31
22    31
23    31
Name: count, dtype: int64

**A join is one of the best data quality instruments you have. Whenever you merge two tables, look at the rows that did not match, and ask whether they have anything in common. If they do, you have learned something about how your data was collected, and it is often something nobody bothered to write down.**

In [67]:
# We can sort our combined dataset
# just like any other dataframe

paired.sort_values('pm25_ugm3', ascending=False).head(10)
# The ten smokiest hours of the month are not one group but two. 
# Four of them are at 01:00, 
# and three of those four sit in the bottom half of the month’s ozone readings...

,datetimeLocal,o3_ppm,pm25_ugm3
464,2024-08-01T01:00:00-07:00,0.011,22.0
142,2024-07-18T01:00:00-07:00,0.022,22.0
619,2024-08-08T01:00:00-07:00,0.025,21.0
673,2024-08-10T10:00:00-07:00,0.029,20.0
609,2024-08-07T15:00:00-07:00,0.043,19.0
585,2024-08-06T13:00:00-07:00,0.040,19.0
349,2024-07-27T01:00:00-07:00,0.008,17.0
293,2024-07-24T15:00:00-07:00,0.027,17.0
295,2024-07-24T17:00:00-07:00,0.026,16.0
586,2024-08-06T14:00:00-07:00,0.041,16.0


In [68]:
print(paired['o3_ppm'].mean())
print(paired.sort_values('pm25_ugm3', ascending=False).head(10)['o3_ppm'].mean())
# The mean of the ten smokiest hours is higher than the mean of every hour,
# 0.029 ppm against 0.022.

0.022438920454545454
0.027200000000000002


### When the columns have different names

In [69]:
url = 'https://eds-217-essential-python.github.io/data/national_parks.csv'
parks = pd.read_csv(url)

parks = parks[parks['year'] != 'Total'].copy()

parks['region'].value_counts()

region
IM    5682
NE    3637
SE    3442
PW    3194
MW    2578
NC    1547
AK    1018
NT      76
Name: count, dtype: int64

In [ ]:
# quickly make a dataframe out of a dictionary
# no csv needed !
regions = pd.DataFrame({
    'code': ['AK', 'IM', 'MW', 'NC', 'NE', 'PW', 'SE'],
    'region_name': ['Alaska', 'Intermountain', 'Midwest', 'National Capital',
                    'Northeast', 'Pacific West', 'Southeast'],
})

regions

,code,region_name
0,AK,Alaska
1,IM,Intermountain
2,MW,Midwest
3,NC,National Capital
4,NE,Northeast
5,PW,Pacific West
6,SE,Southeast


In [72]:
parks.head(1)

,year,gnis_id,geometry,metadata,number_of_records,parkname,region,state,unit_code,unit_name,unit_type,visitors
0,1904,1163670,POLYGON,NaN,1,Crater Lake,PW,OR,CRLA,Crater Lake National Park,National Park,1500.0


In [ ]:
labelled = pd.merge(
    parks, regions, 
    left_on='region', 
    right_on='code',
    how ='inner' # in this case, every region will match a code
    # so no data is lost
    )

labelled[['unit_name', 'region', 'region_name', 'year', 'visitors']].head()
# maps a region name to every region code

,unit_name,region,region_name,year,visitors
0,Crater Lake National Park,PW,Pacific West,1904,1500.0
1,Lake Roosevelt National Recreation Area,PW,Pacific West,1941,0.0
2,Lewis and Clark National Historical Park,PW,Pacific West,1961,69000.0
3,Olympic National Park,PW,Pacific West,1935,2200.0
4,Santa Monica Mountains National Recreation Area,PW,Pacific West,1982,468144.0


In [ ]:
## General pattern
# pd.merge(
#     left, 
#     right, 
#     left_on='column_in_left', 
#     right_on='column_in_right'
#     )

In [84]:
## Check the shapes
print("Are these shapes the same??")
print(parks.shape)
print(labelled.shape)

Are these shapes the same??
(21174, 12)
(21098, 14)


In [78]:
## Did we miss anything ???
checked = pd.merge(parks, regions, left_on='region', right_on='code', how='left')

missing = checked[checked['region_name'].isnull()]

print(missing['region'].value_counts())
print(missing['unit_name'].unique())
print("Oh no!! We forgot to define this one in our dictionary :(")

region
NT    76
Name: count, dtype: int64
['Blue Ridge Parkway']
Oh no!! We forgot to define this one in our dictionary :(


### Key points
The join pattern is pd.merge(left, right, on='key'). The key is the column the two tables have in common, and its values are what pandas matches the rows on.

Use left_on= and right_on= when the key columns have different names in the two tables.

how= decides which rows survive: 'inner' keeps keys present in both (the default), 'left' keeps every row of the left table, 'right' the mirror, and 'outer' keeps everything.

Always compare the row count before and after a merge. No error appears and no warning prints when a join throws data away.

Always run .isnull().sum() after a left, right or outer join. Those three joins exist in order to produce nulls, and counting the nulls tells you how many rows found no match.

The rows a join fails to match are usually not random. Look at them, because they often tell you something about how the data was collected.

Rename your measurement columns before you merge, so you never have to work out what value_x was.

A lookup table you write yourself is a legitimate second table, and often the most useful one you have (all seven rows of it!).